# Preprocesamiento y feature engineering

Cada transformación que se aplica aquí queda justificada por un hallazgo del EDA. No se aplica nada por inercia o convención.

**Autor:** Andrés Fernando Gómez Rojas

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import joblib
from src.data_loader import load_raw
from src.preprocessing import engineer_features, build_feature_matrix, NUM_FEATURES, CAT_FEATURES

pd.set_option('display.float_format', '{:.3f}'.format)

## Carga y feature engineering

In [2]:
df_raw = load_raw('../data/raw/data_customers.csv')
df = engineer_features(df_raw)
df[['n_sessions', 'totals.hits', 'hits_per_session', 'pageviews_per_session',
    'hits_per_pageview', 'part_of_day', 'browser_top', 'os_top']].head()

,n_sessions,totals.hits,hits_per_session,pageviews_per_session,hits_per_pageview,part_of_day,browser_top,os_top
0,1,14.000,14.000,13.000,1.077,noche,Chrome,Macintosh
1,3,14.000,4.667,3.667,1.273,noche,Chrome,Macintosh
2,2,12.500,6.250,5.250,1.190,tarde,Chrome,Macintosh
3,1,22.000,22.000,20.000,1.100,noche,Chrome,Other
4,2,9.500,4.750,4.750,1.000,tarde,Chrome,Other


## Decisiones de transformación aplicadas

**Variables derivadas:**

* `hits_per_session` y `pageviews_per_session`: dividen el total por el número de sesiones del visitante. Permiten distinguir un visitante con 30 hits en una sesión de uno con 30 hits repartidos en 10 sesiones.
* `hits_per_pageview`: proxy de interacción más allá de la simple navegación. Captura clicks en producto, eventos y otras interacciones no contadas como pageview.
* `part_of_day`: discretización de la hora en cuatro bloques (madrugada, mañana, tarde, noche), porque la hora cruda apenas separó nada en el EDA.

**Transformaciones sobre variables sesgadas:**

* Clip al percentil 99 antes del log: evita que cinco o seis outliers extremos definan la escala de toda la variable.
* `log1p`: comprime la cola derecha. Convierte distribuciones con skew > 5 en distribuciones cercanas a normales, lo que beneficia tanto al clustering euclidiano como al StandardScaler.

**Agrupación de categorías raras:**

* `browser_top`: solo Chrome, Safari y Firefox; el resto en Other. Sin esto generaríamos columnas one-hot con 5 o 10 visitantes cada una, que solo añaden ruido al cálculo de distancias.
* `os_top`: solo Macintosh, Windows, iOS y Android; el resto en Other.

**Variables eliminadas:**

* `trafficSource.medium`: Cramer's V = 0.98 con `channelGrouping`. Redundancia perfecta documentada en el EDA.

## Verificación post-transformación

Confirmamos que las variables transformadas tienen distribuciones razonables para el clustering.

In [3]:
from scipy.stats import skew, kurtosis
log_vars = ['log_n_sessions', 'log_totals.hits', 'log_totals.pageviews',
            'log_hits_per_session', 'log_pageviews_per_session']
pd.DataFrame({
    'skew': df[log_vars].apply(skew).round(2),
    'kurtosis': df[log_vars].apply(kurtosis).round(2),
})

,skew,kurtosis
log_n_sessions,0.880,0.310
log_totals.hits,-0.290,-0.160
log_totals.pageviews,-0.300,-0.150
log_hits_per_session,0.050,-1.020
log_pageviews_per_session,0.070,-1.070


Compárense estos valores con los del EDA: skew bajó de >5 a valores cercanos a cero, kurtosis bajó de >60 a valores manejables. Las variables ahora son aptas para clustering basado en distancia euclidiana.

## Construcción de la matriz final

Numéricas escaladas con StandardScaler; categóricas en one-hot. La matriz resultante combina ambas en un solo array.

In [4]:
X, scaler, feature_names = build_feature_matrix(df)
print(f'Shape de X: {X.shape}')
print(f'Total de features: {len(feature_names)}')
print(f'  - numéricas: {len(NUM_FEATURES)}')
print(f'  - one-hot:   {len(feature_names) - len(NUM_FEATURES)}')

Shape de X: (9996, 32)
Total de features: 32
  - numéricas: 8
  - one-hot:   24


## Persistencia

Guardamos la matriz X, el scaler y el DataFrame procesado para que los notebooks posteriores los consuman sin re-ejecutar todo el preprocesamiento.

In [5]:
np.save('../models/X.npy', X)
joblib.dump(scaler, '../models/scaler.pkl')
df.to_pickle('../data/processed/df_processed.pkl')
print('Artefactos guardados en ../models y ../data/processed')

Artefactos guardados en ../models y ../data/processed
